## 1. Instalación e importación de librerías

In [1]:
import os, kagglehub, json, torch
import pandas as pd
from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    size, col, length, lower, trim, regexp_replace,
    when, year, to_date, avg, explode
)

from pyspark.ml.feature import HashingTF, IDF, Tokenizer, StopWordsRemover
from pyspark.sql.types import StringType
from transformers import pipeline
from sentence_transformers import SentenceTransformer

from pyspark.ml.clustering import KMeans


## 2. Creación de la sesión Spark

En esta etapa se inicializa la sesión de Spark, que constituye el punto de entrada al entorno de procesamiento distribuido.

La sesión permite:

- crear y manipular DataFrames distribuidos
- ejecutar transformaciones sobre grandes volúmenes de datos
- aplicar algoritmos de machine learning del ecosistema Spark

In [2]:
spark = SparkSession.builder \
    .appName("NLP_PySpark") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.local.dir", "D:/spark-temp") \
    .getOrCreate()

## 3. Carga y preparación inicial de datos

El dataset es cargado en un DataFrame de PySpark, lo que permite trabajar de forma distribuida con grandes volúmenes de información.

A diferencia de pandas, PySpark permite escalar el procesamiento y aplicar transformaciones de forma eficiente sobre datasets que no cabrían en memoria local.

En esta etapa se seleccionan las columnas relevantes para el análisis textual.

In [3]:
dataset_path = r"D:\repositorios\VSC\COVID_Pyspark\archive" #luego cambiar

metadata_path = os.path.join(dataset_path, "metadata.csv")

df_raw = spark.read.csv(
    metadata_path,
    header=True,
    inferSchema=True
)

## 4. Limpieza y preparación de los datos

En esta sección se prepara el dataset para el análisis NLP.

Las tareas principales incluyen:

- selección de columnas relevantes,
- eliminación de registros incompletos,
- depuración de valores problemáticos.

El objetivo es obtener un DataFrame consistente, enfocado en el contenido textual y listo para su procesamiento posterior en PySpark.

Aqui vamos a seleccionar las columnas más relevantes para nuestro análisis 

In [4]:
df_selected = df_raw.select(
    "cord_uid",
    "title",
    "abstract",
    "journal",
    "authors",
    "url"
)

#### Filtrado de datos relevantes

Se eliminan registros que no contienen información textual clave, como título o abstract.

En el contexto de NLP, la calidad del texto es crítica: documentos incompletos o vacíos no aportan información semántica y pueden afectar negativamente las representaciones vectoriales y los modelos posteriores.


Aquí vamos a excluir las filas que tienen valores perdidos en las columnas `title` o `abstract`. A su vez, vamos a eliminar las filas que aparecen con valores duplicados en la variable `cord_uid`




In [5]:
df_clean = df_selected \
    .filter(col("title").isNotNull()) \
    .filter(col("abstract").isNotNull()) \
    .dropDuplicates(["cord_uid"])

print("Filas después de limpieza:", df_clean.count())

Filas después de limpieza: 764686


In [6]:
df_clean = (
    df_clean
    .withColumn("title_clean", trim(lower(col("title"))))
    .withColumn("abstract_clean", trim(lower(col("abstract"))))
    .withColumn("abstract_clean", regexp_replace(col("abstract_clean"), r"\s+", " "))
    .withColumn("abstract_length", length(col("abstract")))
    .withColumn("title_length", length(col("title")))
)

In [7]:
df_text_ready = (
    df_clean
    .filter(col("abstract_length") > 100)
)

print(f"Rows ready for NLP: {df_text_ready.count()}")

df_text_ready.select("cord_uid", "title", "abstract_length").show(10, truncate=80)

Rows ready for NLP: 759230
+--------+--------------------------------------------------------------------------------+---------------+
|cord_uid|                                                                           title|abstract_length|
+--------+--------------------------------------------------------------------------------+---------------+
|000yfedk|Sleep Disturbances in Frontline Health Care Workers During the COVID-19 Pande...|           2561|
|0012g4t9|COVID-19 and mental health in Brazil: psychiatric symptoms in the general pop...|           1551|
|001lk86f|The Epidemic of Covid-19 in Africa: Demographic Effect, Under-Reporting of Ca...|           2236|
|002y5r4z|Nusinersen in pediatric and adult patients with type III spinal muscular atrophy|           1344|
|0036mel2|Effects of Gender Role Beliefs on Social Connectivity and Marital Safety: Fin...|           2194|
|0056tfa3|Differences in COVID-19 Vaccine Concerns Among Asian Americans and Pacific Is...|           1773|
|

Longitud promedio del abstract

In [8]:
df_text_ready.groupBy("journal") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(20, truncate=80)

+--------------------------------------------+-----+
|                                     journal|count|
+--------------------------------------------+-----+
|                                        NULL|73073|
|                                     bioRxiv| 8942|
|                                    PLoS One| 8873|
|             Int J Environ Res Public Health| 8168|
|                                     Sci Rep| 5281|
|                                      Cureus| 3910|
|                               Front Psychol| 3283|
|                                    BMJ Open| 3276|
|                                     Viruses| 3098|
|                               Front Immunol| 3021|
|                              Sustainability| 2789|
|                         Front Public Health| 2772|
|                               Int J Mol Sci| 2292|
|                         Journal of virology| 2120|
|                                 J Med Virol| 1962|
|                                  J Clin Med|

In [9]:
df_analysis = df_text_ready.select(
    "cord_uid",
    "title",
    "abstract", 
    "abstract_clean",
    "abstract_length", 
    "journal"
)

## 5. Tokenización y eliminación de stopwords

En esta etapa se inicia el procesamiento estructurado del texto dentro del pipeline de NLP.

- **Tokenización (`Tokenizer`)**: el texto de cada abstract es dividido en tokens, es decir, en palabras individuales.  
  Este paso transforma el texto no estructurado en una secuencia de unidades que pueden ser procesadas por algoritmos.  
  La tokenización es fundamental, ya que define cómo se segmenta el lenguaje y afecta directamente las representaciones posteriores.

- **Eliminación de stopwords (`StopWordsRemover`)**: se eliminan palabras muy frecuentes como “the”, “and”, “of”, que aparecen en la mayoría de los documentos pero no aportan información relevante para distinguir temas.  
  Mantener estas palabras puede introducir ruido y reducir la capacidad del modelo para identificar términos realmente informativos.

El resultado es una representación más limpia del texto (`filtered_tokens`), centrada en términos con mayor contenido semántico, lo que mejora la calidad de las etapas posteriores como la vectorización (TF-IDF) y el clustering.

In [10]:
tokenizer = Tokenizer(
    inputCol="abstract_clean",
    outputCol="tokens"
)
df_tokens = tokenizer.transform(df_analysis)

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)
df_tokens = remover.transform(df_tokens)


## Filtrado de documentos sin contenido útil

Después de la eliminación de stopwords, algunos documentos pueden quedar sin tokens, ya que originalmente estaban compuestos únicamente por palabras muy frecuentes y poco informativas.

Estos documentos no aportan valor semántico al análisis y pueden generar problemas en etapas posteriores como la vectorización (TF-IDF) o el clustering.

Por este motivo, se filtran aquellos registros cuyo conjunto de tokens resultante está vacío, asegurando que todos los documentos contengan información relevante para el pipeline de NLP.

In [11]:
df_tokens_filtered = df_tokens.filter(size(col("filtered_tokens")) > 0)

In [12]:
hashingTF = HashingTF(
    inputCol="filtered_tokens",
    outputCol="raw_features",
    numFeatures=1000
)

df_tf = hashingTF.transform(df_tokens_filtered)

idf = IDF(
    inputCol="raw_features",
    outputCol="tfidf_features"
)
idf_model = idf.fit(df_tf)
df_tfidf = idf_model.transform(df_tf)


## Aplicación de KMeans para clustering de documentos

Una vez que los textos han sido transformados en vectores TF-IDF, es posible aplicar algoritmos de machine learning. En este caso, se utiliza KMeans, un algoritmo de clustering no supervisado que agrupa documentos en función de su similitud.

Para facilitar la ejecución:
- se toma una muestra de 1000 documentos (`limit(1000)`) para reducir el costo computacional
- se ajusta el número de particiones (`repartition(2)`) para optimizar el procesamiento en Spark

El modelo se configura con:
- `featuresCol="tfidf_features"`: columna que contiene la representación numérica del texto
- `k=5`: número de clusters a generar
- `seed=42`: para asegurar reproducibilidad de los resultados

Finalmente, el modelo asigna a cada documento un cluster según su similitud con otros documentos.

In [13]:
df_analysis_sample = df_tfidf.limit(1000).repartition(2)

kmeans = KMeans(featuresCol="tfidf_features", predictionCol="cluster", k=5, seed=42)
model = kmeans.fit(df_analysis_sample)

df_clustered = model.transform(df_analysis_sample)
df_clustered.select("title", "cluster").show(10, truncate=80)

+--------------------------------------------------------------------------------+-------+
|                                                                           title|cluster|
+--------------------------------------------------------------------------------+-------+
|Evolving Virulence? Decreasing COVID-19 Complications among Massachusetts Hea...|      0|
|Disentangling the relationship between apolipoprotein E, cardiovascular disea...|      0|
|Systematic review and meta-analysis of the prevalence of common respiratory v...|      1|
|Sleep Disturbances in Frontline Health Care Workers During the COVID-19 Pande...|      1|
|Understanding, Verifying, and Implementing Emergency Use Authorization Molecu...|      0|
|Croatian pupils' perspectives on remote teaching and learning during the covi...|      0|
|Programmed translational frameshifting is likely required for expressions of ...|      0|
|How COVID-19 Transformed Problem-Based Learning at Carle Illinois College of ...|      0|

El resultado muestra cada documento junto con el cluster asignado por el modelo. Cada número de cluster representa un grupo de documentos con contenido similar.

Es importante tener en cuenta que:
- KMeans no conoce el significado del texto, solo agrupa en base a patrones numéricos
- los clusters no tienen una etiqueta explícita, deben interpretarse analizando los textos dentro de cada grupo
- documentos con temas similares tenderán a estar en el mismo cluster

Para interpretar correctamente los resultados, se recomienda:
- revisar varios títulos o abstracts dentro de cada cluster
- identificar temas comunes (por ejemplo: COVID-19, salud mental, educación, etc.)
- analizar si los grupos tienen coherencia temática

Este paso es clave para validar si el modelo está capturando correctamente la estructura del corpus.

## Análisis de términos por cluster

Para interpretar los clusters generados por KMeans, es necesario analizar qué palabras caracterizan a cada grupo. Dado que el modelo trabaja con representaciones numéricas (TF-IDF), no es posible interpretar directamente los clusters sin volver al texto.

En esta etapa se realiza el siguiente proceso:
- se toma una muestra de documentos para facilitar el análisis
- se separan los tokens de cada documento utilizando `explode`
- se agrupan las palabras por cluster
- se calcula la frecuencia de aparición de cada término dentro de cada cluster

Este procedimiento permite identificar los términos más representativos de cada grupo y facilita la interpretación temática de los clusters.

In [14]:
df_cluster_sample = df_clustered.limit(50)

df_cluster_words = (
    df_cluster_sample
    .select("cluster", explode(col("filtered_tokens")).alias("word"))
)

df_cluster_word_counts = (
    df_cluster_words
    .groupBy("cluster", "word")
    .count()
    .orderBy("cluster", col("count").desc())
)


In [15]:
df_clustered.select(
    "cord_uid", "title", "abstract_clean", "journal", "cluster", "abstract_length"
).toPandas().to_csv("clustered_docs.csv", index=False)

In [16]:
from pyspark.sql.functions import avg

df_clustered.groupBy("cluster").agg(
    avg("abstract_length").alias("avg_abstract_length")
).toPandas().to_csv("cluster_stats.csv", index=False)

In [17]:
spark.stop()